## Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Set Project Path

In [2]:
import os

PROJECT_PATH = "/content/drive/MyDrive/rocket_telemetry_project"

os.makedirs(PROJECT_PATH, exist_ok=True)
os.chdir(PROJECT_PATH)

print("Current working directory:", os.getcwd())

Current working directory: /content/drive/MyDrive/rocket_telemetry_project


## Create Required Folders

In [3]:
folders = [
    "models",
    "results",
    "figures"
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("Project folders ready.")

Project folders ready.


## Imports

In [4]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dropout,
    RepeatVector,
    TimeDistributed,
    Dense
)

from tensorflow.keras.optimizers import Adam

## Model Builder Function

In [5]:
def build_lstm_autoencoder(
        timesteps,
        n_features,
        latent_dim=32,
        dropout_rate=0.2):

    inputs = Input(shape=(timesteps, n_features))

    # Encoder
    x = LSTM(128, activation='tanh', return_sequences=True)(inputs)
    x = Dropout(dropout_rate)(x)

    x = LSTM(64, activation='tanh', return_sequences=True)(x)
    x = Dropout(dropout_rate)(x)

    encoded = LSTM(latent_dim, activation='tanh', return_sequences=False)(x)

    # Bridge
    x = RepeatVector(timesteps)(encoded)

    # Decoder
    x = LSTM(latent_dim, activation='tanh', return_sequences=True)(x)
    x = Dropout(dropout_rate)(x)

    x = LSTM(64, activation='tanh', return_sequences=True)(x)
    x = Dropout(dropout_rate)(x)

    x = LSTM(128, activation='tanh', return_sequences=True)(x)

    outputs = TimeDistributed(Dense(n_features))(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="mse"
    )

    assert model.output_shape == model.input_shape

    return model

## Model Configuration Comparison

In [6]:
timesteps = 50
n_features = 75

configs = {
    "small":  {"latent_dim":16, "dropout":0.1},
    "medium": {"latent_dim":32, "dropout":0.2},
    "large":  {"latent_dim":64, "dropout":0.3}
}

for name, cfg in configs.items():

    model = build_lstm_autoencoder(
        timesteps,
        n_features,
        latent_dim=cfg["latent_dim"],
        dropout_rate=cfg["dropout"]
    )

    params = model.count_params()

    memory_mb = params * 4 / (1024**2)

    print(f"\n{name.upper()} MODEL")
    print(f"Latent Dim: {cfg['latent_dim']}")
    print(f"Dropout: {cfg['dropout']}")
    print(f"Parameters: {params:,}")
    print(f"Estimated Memory: {memory_mb:.2f} MB")


SMALL MODEL
Latent Dim: 16
Dropout: 0.1
Parameters: 290,379
Estimated Memory: 1.11 MB

MEDIUM MODEL
Latent Dim: 32
Dropout: 0.2
Parameters: 307,915
Estimated Memory: 1.17 MB

LARGE MODEL
Latent Dim: 64
Dropout: 0.3
Parameters: 361,419
Estimated Memory: 1.38 MB


## Build Medium Model

In [7]:
model = build_lstm_autoencoder(
    timesteps=50,
    n_features=75,
    latent_dim=32,
    dropout_rate=0.2
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 50, 75)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_18 (LSTM)                  │ (None, 50, 128)        │       104,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_19 (LSTM)                  │ (None, 50, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 50, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_20 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_3 (RepeatVector)  │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_21 (LSTM)                  │ (None, 50, 32)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_22 (LSTM)                  │ (None, 50, 64)         │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 50, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_23 (LSTM)                  │ (None, 50, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 50, 75)         │         9,675 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 307,915 (1.17 MB)

 Trainable params: 307,915 (1.17 MB)

 Non-trainable params: 0 (0.00 B)

## Verification

In [8]:
dummy = np.random.randn(5,50,75)

out = model.predict(dummy)

assert out.shape == dummy.shape

print("OUTPUT SHAPE MATCHES INPUT — AUTOENCODER VERIFIED")

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
OUTPUT SHAPE MATCHES INPUT — AUTOENCODER VERIFIED


## Save Model Architecture

In [9]:
import json

architecture = model.to_json()

with open("models/architecture_P1.json", "w") as f:
    f.write(architecture)

print("Architecture saved to models/architecture_P1.json")

Architecture saved to models/architecture_P1.json


## Print Layer Table

In [17]:
print("{:<25} {:<25}".format("Layer", "Output Shape"))
print("-"*55)

for layer in model.layers:
    try:
        shape = layer.output.shape
    except:
        shape = "N/A"

    print("{:<25} {:<25}".format(layer.name, str(shape)))

Layer                     Output Shape             
-------------------------------------------------------
input_layer_3             (None, 50, 75)           
lstm_18                   (None, 50, 128)          
dropout_12                (None, 50, 128)          
lstm_19                   (None, 50, 64)           
dropout_13                (None, 50, 64)           
lstm_20                   (None, 32)               
repeat_vector_3           (None, 50, 32)           
lstm_21                   (None, 50, 32)           
dropout_14                (None, 50, 32)           
lstm_22                   (None, 50, 64)           
dropout_15                (None, 50, 64)           
lstm_23                   (None, 50, 128)          
time_distributed_3        (None, 50, 75)           
